In [ ]:
# 04_ramp_shock_baseline.ipynb -- LightGBM baseline for the 1-4-slot-lead-time
# ramp-shock target (same feature pipeline as 03_violation_baseline.ipynb)
# !pip install lightgbm -q

import sys
sys.path.insert(0, "..")
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import average_precision_score, f1_score, precision_recall_curve

import features as f

TARGET = "ramp_lead"

scada = pd.read_csv("data/study2_scada.csv", parse_dates=["date"])
scada = scada.sort_values(["date", "time"]).reset_index(drop=True)
feat_df = f.build_feature_table(scada)

df = feat_df.dropna(subset=[TARGET]).copy()

# --- Time-aware split (same as 03_violation_baseline.ipynb) ---
train = df[(df["date"] >= "2024-11-04") & (df["date"] <= "2025-06-30")]
val = df[(df["date"] >= "2025-07-01") & (df["date"] <= "2025-12-31")]
test = df[df["date"] >= "2026-01-01"]

print("train:", train.shape, "val:", val.shape, "test:", test.shape)
print("event rate train/val/test:", train[TARGET].mean(), val[TARGET].mean(), test[TARGET].mean())

X_train, y_train = train[f.FEATURE_COLS], train[TARGET]
X_val, y_val = val[f.FEATURE_COLS], val[TARGET]
X_test, y_test = test[f.FEATURE_COLS], test[TARGET]

# NOTE (2026-07-11): NOT using scale_pos_weight -- see features.py's
# scale_pos_weight() docstring and 03_violation_baseline.ipynb for the full story
# (it caused catastrophic 1-round early stopping on the violation target; the
# effect here is milder since ramp_lead's positive rate is much less extreme, but
# removing it still gave a small, consistent improvement across every metric).
model = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, random_state=42, verbosity=-1)
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="average_precision",
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(50)],
)
print(f"\nbest_iteration_: {model.best_iteration_}")

proba_test = model.predict_proba(X_test)[:, 1]
pr_auc = average_precision_score(y_test, proba_test)
base_rate = y_test.mean()
print(f"PR-AUC: {pr_auc:.4f}  (random baseline = base rate = {base_rate:.4f})")

preds_05 = (proba_test >= 0.5).astype(int)
print(f"F1 @ 0.5 threshold: {f1_score(y_test, preds_05):.4f}")

precision, recall, thresh = precision_recall_curve(y_test, proba_test)
f1s = 2 * precision * recall / (precision + recall + 1e-12)
best_idx = np.nanargmax(f1s[:-1])
print(f"Best-F1 operating point: F1={f1s[best_idx]:.4f} at threshold={thresh[best_idx]:.4f} "
      f"(precision={precision[best_idx]:.4f}, recall={recall[best_idx]:.4f})")

idx95 = np.where(precision[:-1] >= 0.95)[0]
recall_at_95p = recall[idx95].max() if len(idx95) else 0.0
print(f"Recall at >=95% precision: {recall_at_95p:.4f}")

importance = pd.Series(model.feature_importances_, index=f.FEATURE_COLS).sort_values(ascending=False)
print("\nTop 15 features:\n", importance.head(15))

# --- Results (verified 2026-07-11, LightGBM 4.6.0; corrected after removing
#     scale_pos_weight and adding two solar-volatility features -- these numbers
#     supersede the first version of this notebook, PR-AUC 0.7140 / F1@0.5 0.6293 /
#     recall@95%P 0.1771) ---
# PR-AUC 0.7248 vs a random/base-rate baseline of 0.1793 -- 4.04x lift over chance
# (was 4x before, a small gain since this target was already strong). F1@0.5 dropped
# to 0.5436 (was 0.6293) -- expected, not a regression: scale_pos_weight artificially
# recalibrates predicted probabilities around the 0.5 boundary, so removing it makes
# a naive 0.5 threshold less meaningful, but the threshold-optimized number is what
# actually matters here (same reasoning 03_violation_baseline.ipynb uses). Best-F1
# operating point (threshold ~0.20): F1=0.6570 (was 0.6293), precision=63.4%,
# recall=68.1%. Recall at >=95% precision = 0.2238 (was 0.1771) -- even in a
# high-confidence-only alert mode, the model now catches ~22% of upcoming ramp-shocks.
#
# Top features are dominated by time-of-day ("hour") and the slot's own recent demand
# trajectory (demand_delta_mw, demand_met_mw_lag1/lag3), consistent with 01_eda.ipynb's
# sunrise/sunset clustering finding -- but solar_delta_mw and solar_roll8_std (added
# 2026-07-11) now place 3rd and 4th, ahead of most demand-lag features. These weren't in
# the original feature set; they were added after an explicit diagnostic (run for
# 03_violation_baseline.ipynb, not this notebook) found solar-generation volatility was
# a better-supported signal than demand-side ramps for the violation target -- and it
# turned out to help THIS target too, materially. Corridor columns (ir_er_wr_net_mu,
# xb_net_nepal_mu, xb_net_bangladesh_mu, ir_wr_nr_net_mu, ir_er_nr_net_mu) still place
# in the top 15 -- weaker than the time/demand/solar signal but a real, non-trivial
# contribution, consistent with Era 2's daily-resolution finding that corridor flow has
# a genuine relationship with grid stress even though it isn't the dominant driver.
#
# This target remains markedly easier to predict with lead time than frequency
# violation (see 03_violation_baseline.ipynb, PR-AUC 0.0937 even after its own fixes)
# -- plausibly because a ramp-shock is a direct, mechanical property of the
# demand/generation trajectory itself, while a frequency violation is a downstream
# consequence that depends on how well AGC/reserves absorb a given ramp, adding a layer
# of noise the raw features here don't fully capture.
